# Read H-Reflex App Data Files

This notebook reads and visualizes data from the **H-Reflex Behavior App** (hreflex_txbdc) binary data files.

**File convention (V2/V3):**
- **`.hrs1`** — MH Recruitment Curve stage: sweeps across stimulation intensities to map the M/H-wave recruitment curve.
- **`.hrs2`** — Control Mode stage: stimulates at a fixed user-set intensity (can be changed between trials).
- **`.hrs3`** — Down Condition Pellet (DCP) stage: closed-loop H-reflex conditioning with pellet reward.
- **`.hrs4`** — Up Condition Pellet stage.
- **`.hrs5`** — Down Condition VNS stage.
- **`.hrs6`** — Up Condition VNS stage.
- **`.hrsft`** — Frequency Test stage.

All trial files share the MhRecHeader + MhRecTrial binary format.
EMG data blocks (raw differential, filtered, abs-value) are embedded in every file.

The binary format is based on the `FileIO_Helpers` serialization from the `hreflex_txbdc` package.

# Section 1: Binary File Reader Utilities

In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from helpers import (
    # File readers
    read_hrs2, read_hrs3, read_hrs4, read_hrs5, read_hrs6, read_hrs_ft,
    find_hrs_files, find_all_hrs_ft_files, detect_app_version,
    # File writer
    write_hrs2,
    # Summary printers
    print_hrs2_summary,
    # HRS2 plots
    plot_amplitude_distribution, plot_background_emg_views, plot_hrs2_analysis,
    plot_actual_trial_timeline, plot_mwave_control_error, plot_frequency_test,
    # Frequency Test analysis plots
    plot_ft_depression_curve, plot_ft_averaged_waveforms, plot_ft_peak_curve,
    # SNR analysis
    compute_snr_analysis, compute_mra_snr_analysis,
    plot_hrs2_trials, classify_trials, get_trial_window,
    detect_stim_onset, get_trial_context_window,
    detect_and_correct_failed_trials,
    # Post-hoc global windowing analysis
    analyze_global_background, run_threshold_sweep, plot_threshold_sweep,
    split_trials_by_polarity, plot_hm_ratio_summary, plot_hwave_regression,
    # Constants
    SAMPLE_RATE, BIN_DURATION_MS, BIN_SAMPLES, TRIAL_RECORD_MS,
    STIM_ONSET_THRESHOLD, STIM_END_THRESHOLD,
    build_merged_amp_groups,
    make_viewer, compute_h_comparison_data, plot_h_reflex_comparison,
    FT_SNAP_HZ, ft_snap_hz, ft_stim_adc_hz, compute_ft_trial_hz,
    load_all_recordings, make_ft_viewer,
)

print("Helpers loaded.")
print(f"  App constants: SAMPLE_RATE={SAMPLE_RATE} Hz | BIN={BIN_DURATION_MS} ms ({BIN_SAMPLES} samples) | TRIAL_RECORD={TRIAL_RECORD_MS} ms")
print(f"  Stim thresholds: onset >= {STIM_ONSET_THRESHOLD} V | end < {STIM_END_THRESHOLD} V")

Helpers loaded.
  App constants: SAMPLE_RATE=5000.0 Hz | BIN=50 ms (250 samples) | TRIAL_RECORD=100 ms
  Stim thresholds: onset >= 4.5 V | end < 1.9 V


# Section 1b: Auto-Detect Recording Files

Set `recording_dir` to the path of your recording folder.  
The `.hrs1` and `.hrs2` files will be found automatically.

In [4]:
# ── Multi-Recording Configuration ─────────────────────────────────────────────
# Each entry: ("Display Label", "relative/path/to/recording_dir", sample_rate_hz)
# sample_rate_hz: explicit Hz (e.g. 10000.0 or 5000.0); None = auto-detect from data.
RECORDING_DIRS = [

    ("HRPILOT-17 ControlM5",  "HRPilot-17_Control/BASELINE5_HRPILOT-17_BOOTH1_250US_10KHZ_8-26-26",  10000.0),

    # Add more recordings below — uncomment or append new tuples.
]
# When multiple recordings are loaded a Recording dropdown appears in the selector cell.
print(f"{len(RECORDING_DIRS)} recording(s) configured.")
for _i, (_lbl, _rdir, _rsr) in enumerate(RECORDING_DIRS):
    _sr_str = f"{_rsr} Hz" if _rsr else "auto-detect"
    print(f"  [{_i}] {_lbl!r}  →  {_rdir}  (sample_rate={_sr_str})")

1 recording(s) configured.
  [0] 'HRPILOT-17 ControlM5'  →  HRPilot-17_Control/BASELINE5_HRPILOT-17_BOOTH1_250US_10KHZ_8-26-26  (sample_rate=10000.0 Hz)


In [6]:
# ── Load all recordings (auto-detects V2 / V3) ─────────────────────────────────
# Customize FT_SNAP_HZ to match your experiment's pulse-train frequencies (Hz).
# Values within ±25% of a target snap to that target.
FT_SNAP_HZ_CUSTOM = FT_SNAP_HZ  # use [5.0, 10.0, 15.0, 20.0, 33.0] or override here

_all_recordings = load_all_recordings(RECORDING_DIRS, ft_snap_hz_list=FT_SNAP_HZ_CUSTOM)
_active_rec_label = next(iter(_all_recordings))
print(f'\n{len(_all_recordings)} recording(s) loaded.  Active: {_active_rec_label!r}')


── Loading: 'HRPILOT-17 ControlM5'  (HRPilot-17_Control/BASELINE5_HRPILOT-17_BOOTH1_250US_10KHZ_8-26-26)
   .hrs1: not found
   .hrs2: 1026 trials  (Control Mode)
   Note: Control Mode aliased as primary analysis (no MH Recruitment stage).
   App V3  |  Stages: ['control_mode']  |  SR: 10000.0 Hz

1 recording(s) loaded.  Active: 'HRPILOT-17 ControlM5'


# Section 3: Peri-Stimulus Trials — MH Recruitment Curve or Control Mode

`hrs2_trials` contains whichever stage was run:
- `.hrs1` present → MH Recruitment Curve trials
- `.hrs1` absent, `.hrs2` present → Control Mode trials (aliased automatically)

In [7]:
# Stage summary for the active recording
print(f'Active recording: {_active_rec_label!r}')
_rec_info = _all_recordings[_active_rec_label]
for _sk, (_st, _sh, _se, _slbl) in _rec_info['stage_map'].items():
    print(f'  {_slbl}: {len(_st)} trials')
print(f'\n(Run the Stage Selector cell below to enable the interactive dropdown.)')

Active recording: 'HRPILOT-17 ControlM5'
  Control Mode (.hrs2): 1026 trials

(Run the Stage Selector cell below to enable the interactive dropdown.)


In [8]:
# ── Recording & Stage Viewer Factory ───────────────────────────────────────────
# Each viewer section below has its own independent Recording + Stage dropdowns.
# Set each viewer to a different recording/stage to compare them side by side.
# make_viewer() is defined in helpers.py
from IPython.display import display as _disp

# ── Loaded recordings summary ────────────────────────────────────────────────────
print(f'{len(_all_recordings)} recording(s) loaded:')
for _rl, _rd in _all_recordings.items():
    print(f'  {_rl!r}  (App V{_rd["app_version"]}  |  {_rd["sample_rate"]} Hz)')
    for _sk, (_st, _sh, _se, _slbl) in _rd['stage_map'].items():
        print(f'    · {_slbl}: {len(_st)} trials')
print()
print('Each viewer below has its own Recording + Stage dropdowns for independent selection.')

# ── Stimulation Intensity Histogram ─────────────────────────────────────────────────────
def _render_hist(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
    print(f'\n── Histogram: {stage_label}  ({len(trials)} trials)  [{rec_label}]')
    plot_amplitude_distribution(trials, header)

_hist_widget, _hist_render = make_viewer(_all_recordings, _active_rec_label, _render_hist)
_disp(_hist_widget)
_hist_render()

1 recording(s) loaded:
  'HRPILOT-17 ControlM5'  (App V3  |  10000.0 Hz)
    · Control Mode (.hrs2): 1026 trials

Each viewer below has its own Recording + Stage dropdowns for independent selection.


# Section 3b: "Most Recent Background" + "Background EMG Level"

Recreates the H-Reflex App recruitment-curve trial-plot widgets from `MhRecruitmentCurveStage.get_trial_plot_options`:

- **Most recent background** bar chart of the pre-stim |EMG| bins.
- **EMG Level** scatter of background grand means across trials.

Bins are reconstructed from `hrs2_emg_blocks` over a fixed monitoring window (default 2500 ms ending at trigger time).

In [9]:
# ── Background EMG Views ──────────────────────────────────────────────────────
from IPython.display import display as _disp

def _render_bg(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
    print(f'\n── Background EMG: {stage_label}  ({len(trials)} trials)  [{rec_label}]')
    plot_background_emg_views(trials, emg_blocks, monitoring_window_ms=2500)

_bg_widget, _bg_render = make_viewer(_all_recordings, _active_rec_label, _render_bg)
_disp(_bg_widget)
_bg_render()

# Section 6: HRS2 Detailed Analysis

Interactive averaged-waveform paged grid (with M/H peak markers and signal overlays) plus the normalized and raw recruitment curves. The cell below sets the analysis parameters; the cell after that calls `plot_hrs2_analysis`.

In [10]:
#  Configuration 
PRE_PLOT_MS  = 5   # ms before stim onset to display
POST_PLOT_MS = 25  # ms after  stim onset to display
N_PER_PAGE   = 6   # trials shown per page (2 rows x 3 cols)

# M/H wave window constants (ms relative to stim onset)
#M_WAVE_START_MS = 3
#M_WAVE_END_MS= 5.5
#H_WAVE_START_MS = 9.0
#H_WAVE_END_MS   = 12

M_WAVE_START_MS = 2.2 
M_WAVE_END_MS= 4.2
H_WAVE_START_MS = 6   
H_WAVE_END_MS   = 9.6

PRE_AVG_MS  = 5   # ms before stim onset
POST_AVG_MS = 25  # ms after  stim onset
N_PER_PAGE  = 6   # amplitude groups per page (2 rows x 3 cols)


In [11]:
# ── Pre-compute Comparison Data ───────────────────────────────────────────────────────
# Computed once here (after configuration constants are set) for instant rendering
# in the comparison plot below. Re-run this cell if you change PRE_AVG_MS,
# POST_AVG_MS, M_WAVE_START_MS, M_WAVE_END_MS, H_WAVE_START_MS, or H_WAVE_END_MS.
_xr_cache = compute_h_comparison_data(
    _all_recordings, PRE_AVG_MS, POST_AVG_MS,
    H_WAVE_START_MS, H_WAVE_END_MS,
    M_WAVE_START_MS, M_WAVE_END_MS,
)
_xr_stages = {}
for _rd in _all_recordings.values():
    for _sk, (_st, _sh, _se, _slbl) in _rd['stage_map'].items():
        if _st and _sk not in _xr_stages:
            _xr_stages[_sk] = _slbl
print(f'Comparison data pre-computed for {len(_xr_cache)} recording(s), '
      f'{len(_xr_stages)} stage(s): {list(_xr_stages.keys())}')

Comparison data pre-computed for 1 recording(s), 1 stage(s): ['control_mode']


In [12]:
# ── Optional: Merged Amplitude Group Analysis ─────────────────────────────────
# MERGE_ALL = True  →  collapse every amplitude into a single group.
#
# MERGED_GROUPS accepts two styles (mix freely):
#   Explicit list : [0.12, 0.13, 0.15]   — merge exactly those amplitudes
#   Range tuple   : (0.10, 0.50)         — merge all amplitudes where low <= amp <= high
#
# Examples:
#   MERGED_GROUPS = [[0.12, 0.13], [0.15, 0.16, 0.17]]   ← two explicit groups
#   MERGED_GROUPS = [(0.10, 0.20), (0.25, 0.40)]          ← two range groups
#   MERGED_GROUPS = [(0.10, 0.20), [0.50, 0.55]]          ← range + explicit, mixed
#
# Leave both flags at their defaults to use the standard (unmerged) grouping.
MERGE_ALL     = False   # True → collapse every amplitude into one group
MERGED_GROUPS = []
#[(0,0.108), (0.108,0.23), (0.23,0.43), (0.43,0.50), (0.50,0.60), (0.60,0.70)]

def _resolve_groups(trials, groups):
    _all_amps = sorted({t.stimulation_amplitude_ma for t in trials})
    resolved = []
    for g in groups:
        if isinstance(g, tuple) and len(g) == 2:
            lo, hi = float(g[0]), float(g[1])
            matched = [a for a in _all_amps if lo <= a <= hi]
            if matched:
                resolved.append(matched)
            else:
                print(f'Warning: range ({lo}, {hi}) matched no amplitudes — skipped.')
        else:
            resolved.append(list(g))
    return resolved

def _apply_merge(trials):
    if MERGE_ALL:
        _amps = sorted({t.stimulation_amplitude_ma for t in trials})
        return build_merged_amp_groups(trials, [_amps])
    if MERGED_GROUPS:
        return build_merged_amp_groups(trials, _resolve_groups(trials, MERGED_GROUPS))
    return trials

print("Merge config: MERGE_ALL =", MERGE_ALL, " | MERGED_GROUPS =", MERGED_GROUPS or "(none)")
print("Re-run viewer cells or change Recording/Stage to apply new merge settings.")

Merge config: MERGE_ALL = False  | MERGED_GROUPS = (none)
Re-run viewer cells or change Recording/Stage to apply new merge settings.


In [13]:
# ── HRS2 Analysis: Interactive Averaged Waveforms + Recruitment Curve ─────────
from IPython.display import display as _disp

def _render_ana(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
    tp = _apply_merge(trials)
    print(f'\n── Analysis: {stage_label}  ({len(trials)} trials)  [{rec_label}]')
    plot_hrs2_analysis(
        tp, header,
        pre_avg_ms=PRE_AVG_MS, post_avg_ms=POST_AVG_MS,
        n_per_page=N_PER_PAGE,
        m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
        h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
        sample_rate=sr or h1h.sample_rate,
        emg_blocks=emg_blocks,
    )

_ana_widget, _ana_render = make_viewer(_all_recordings, _active_rec_label, _render_ana)
_disp(_ana_widget)
_ana_render()

In [14]:
# ── HRS2 Trial Viewer: Interactive Per-Trial Grid + Zoom ──────────────────────
from IPython.display import display as _disp

def _render_trv(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
    tp = _apply_merge(trials)
    print(f'\n── Trial Viewer: {stage_label}  ({len(trials)} trials)  [{rec_label}]')
    plot_hrs2_trials(
        tp, header,
        pre_plot_ms=PRE_PLOT_MS, post_plot_ms=POST_PLOT_MS,
        n_per_page=N_PER_PAGE,
        m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
        h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
        sample_rate=sr or h1h.sample_rate,
        emg_blocks=emg_blocks,
    )

_trv_widget, _trv_render = make_viewer(_all_recordings, _active_rec_label, _render_trv)
_disp(_trv_widget)
_trv_render()

# Section 7: Trial Cleaning — Exclude Bad Trials & Export

Filter out unwanted trials using one or more of the methods below, preview the result, then write the cleaned data to a new binary file in the same format as the original.

**Available filter methods (combine freely — a trial is excluded if ANY rule rejects it):**
- **Method 1 — Trial number**: exclude specific trial numbers (1-based) or ranges
- **Method 2 — M-wave MRA**: keep only trials where the stored M-wave MRA falls within [M_MRA_MIN, M_MRA_MAX]
- **Method 3 — H-wave MRA**: keep only trials where the stored H-wave MRA falls within [H_MRA_MIN, H_MRA_MAX]
- **Method 4 — Stimulation amplitude**: keep only trials within [STIM_MIN_MA, STIM_MAX_MA]

In [15]:
# ── Trial Cleaning Configuration ────────────────────────────────────────────────
# Select which recording and stage to clean.
RECORDING_TO_CLEAN = _active_rec_label   # or a specific label from RECORDING_DIRS
STAGE_TO_CLEAN     = list(_all_recordings[_active_rec_label]['stage_map'].keys())[0]
                    # e.g. 'control_mode', 'mh_recruitment', 'dcp' — see stage summary above

# ── Method 1: Exclude by trial number (1-based) ─────────────────────────────────
EXCLUDE_TRIAL_NUMS   = []           # e.g. [3, 7, 12]       — individual trial numbers
EXCLUDE_TRIAL_RANGES = [(643,777)]           # e.g. [(1, 5), (20, 30)] — inclusive ranges

# ── Method 2: M-wave MRA window (µV) — None = no limit ─────────────────────────
M_MRA_MIN = None     # keep trials with M-wave MRA >= this
M_MRA_MAX = None     # keep trials with M-wave MRA <= this

# ── Method 3: H-wave MRA window (µV) — None = no limit ─────────────────────────
H_MRA_MIN = None
H_MRA_MAX = None

# ── Method 4: Stimulation amplitude window (mA) — None = no limit ──────────────
STIM_MIN_MA = None
STIM_MAX_MA = None

print(f"Will clean: {RECORDING_TO_CLEAN!r} → stage {STAGE_TO_CLEAN!r}")
_rec_clean = _all_recordings[RECORDING_TO_CLEAN]
_st_clean, _sh_clean, _se_clean, _slbl_clean = _rec_clean['stage_map'][STAGE_TO_CLEAN]
print(f"Total trials in stage: {len(_st_clean)}")

Will clean: 'HRPILOT-17 ControlM5' → stage 'control_mode'
Total trials in stage: 1026


In [16]:
# ── Apply Filters — Preview ─────────────────────────────────────────────────────
# Run this cell to see exactly which trials will be kept and excluded.
# Nothing is written yet.

_excl_num_set = set(EXCLUDE_TRIAL_NUMS)

def _trial_exclusion_reason(trial, trial_num_1based):
    """Return a reason string if the trial should be excluded, else ''."""
    reasons = []

    # Method 1: trial number
    if trial_num_1based in _excl_num_set:
        reasons.append(f'number {trial_num_1based} in EXCLUDE_TRIAL_NUMS')
    for lo, hi in EXCLUDE_TRIAL_RANGES:
        if lo <= trial_num_1based <= hi:
            reasons.append(f'number {trial_num_1based} in range ({lo}–{hi})')

    # Method 2: M-wave MRA
    m_mra = getattr(trial, 'm_wave_mra', None)
    if m_mra is not None:
        if M_MRA_MIN is not None and m_mra < M_MRA_MIN:
            reasons.append(f'M-MRA {m_mra:.2f} < {M_MRA_MIN}')
        if M_MRA_MAX is not None and m_mra > M_MRA_MAX:
            reasons.append(f'M-MRA {m_mra:.2f} > {M_MRA_MAX}')

    # Method 3: H-wave MRA
    h_mra = getattr(trial, 'h_wave_mra', None)
    if h_mra is not None:
        if H_MRA_MIN is not None and h_mra < H_MRA_MIN:
            reasons.append(f'H-MRA {h_mra:.2f} < {H_MRA_MIN}')
        if H_MRA_MAX is not None and h_mra > H_MRA_MAX:
            reasons.append(f'H-MRA {h_mra:.2f} > {H_MRA_MAX}')

    # Method 4: stim amplitude
    amp = getattr(trial, 'stimulation_amplitude_ma', None)
    if amp is not None:
        if STIM_MIN_MA is not None and amp < STIM_MIN_MA:
            reasons.append(f'stim {amp:.3f} mA < {STIM_MIN_MA}')
        if STIM_MAX_MA is not None and amp > STIM_MAX_MA:
            reasons.append(f'stim {amp:.3f} mA > {STIM_MAX_MA}')

    return '; '.join(reasons)


def apply_trial_filters(trials, emg_blocks):
    """Return (kept_trials, kept_emg, excluded_info).
    excluded_info: list of (trial_num_1based, trial, reason_str)
    """
    kept_t, kept_e, excl = [], [], []
    for i, t in enumerate(trials):
        num = i + 1
        reason = _trial_exclusion_reason(t, num)
        if reason:
            excl.append((num, t, reason))
        else:
            kept_t.append(t)
            if emg_blocks and i < len(emg_blocks):
                kept_e.append(emg_blocks[i])
    return kept_t, kept_e, excl


_kept_trials, _kept_emg, _excluded_info = apply_trial_filters(_st_clean, _se_clean)

print(f"Recording : {RECORDING_TO_CLEAN!r}")
print(f"Stage     : {_slbl_clean}")
print(f"Total     : {len(_st_clean)} trials")
print(f"Keep      : {len(_kept_trials)} trials")
print(f"Exclude   : {len(_excluded_info)} trials")

if _excluded_info:
    print(f"\n{'Trial':>6}  {'Stim (mA)':>10}  {'M-MRA (µV)':>12}  {'H-MRA (µV)':>12}  Reason")
    print("─" * 80)
    for num, t, reason in _excluded_info:
        amp   = getattr(t, 'stimulation_amplitude_ma', float('nan'))
        m_mra = getattr(t, 'm_wave_mra',               float('nan'))
        h_mra = getattr(t, 'h_wave_mra',               float('nan'))
        print(f"{num:>6}  {amp:>10.3f}  {m_mra:>12.2f}  {h_mra:>12.2f}  {reason}")
else:
    print("\nNo trials excluded — all pass the current filter criteria.")

Recording : 'HRPILOT-17 ControlM5'
Stage     : Control Mode (.hrs2)
Total     : 1026 trials
Keep      : 891 trials
Exclude   : 135 trials

 Trial   Stim (mA)    M-MRA (µV)    H-MRA (µV)  Reason
────────────────────────────────────────────────────────────────────────────────
   643       0.140           nan           nan  number 643 in range (643–777)
   644       0.140           nan           nan  number 644 in range (643–777)
   645       0.140           nan           nan  number 645 in range (643–777)
   646       0.140           nan           nan  number 646 in range (643–777)
   647       0.140           nan           nan  number 647 in range (643–777)
   648       0.140           nan           nan  number 648 in range (643–777)
   649       0.140           nan           nan  number 649 in range (643–777)
   650       0.140           nan           nan  number 650 in range (643–777)
   651       0.140           nan           nan  number 651 in range (643–777)
   652       0.140     

In [17]:
# ── Write Cleaned File ─────────────────────────────────────────────────────────
# Run AFTER reviewing the preview above.
# The output file is in the exact same binary format as the input.
# Default: saves a new file in the same directory with '_cleaned' appended to the stem.

import os as _os

# Find the original file path for the selected stage
_rdir_clean = next(d for lbl, d, _ in RECORDING_DIRS if lbl == RECORDING_TO_CLEAN)
_rp1c, _rp2c, _rp3c, _rp4c, _rp5c, _rp6c, _rpftc = find_hrs_files(_rdir_clean)
_stage_src_path = {
    'mh_recruitment': _rp1c,
    'control_mode':   _rp2c,
    'dcp':            _rp3c,
    'up_cond_pellet': _rp4c,
    'down_cond_vns':  _rp5c,
    'up_cond_vns':    _rp6c,
}[STAGE_TO_CLEAN]

if _stage_src_path is None:
    print(f"ERROR: source file for stage {STAGE_TO_CLEAN!r} not found in {_rdir_clean!r}")
else:
    _stem, _ext = _os.path.splitext(_stage_src_path)
    OUTPUT_PATH = _stem + '_cleaned' + _ext   # ← change this to override the output path

    if len(_kept_trials) == 0:
        print("WARNING: 0 trials kept — nothing to write. Adjust your filters.")
    elif _os.path.abspath(OUTPUT_PATH) == _os.path.abspath(_stage_src_path):
        print("ERROR: OUTPUT_PATH is the same as the source file. Change OUTPUT_PATH to avoid overwriting.")
    else:
        write_hrs2(OUTPUT_PATH, _sh_clean, _kept_trials, _kept_emg)
        print(f"Wrote {len(_kept_trials)} / {len(_st_clean)} trials to:")
        print(f"  {OUTPUT_PATH}")
        print(f"\nExcluded {len(_excluded_info)} trial(s). Original file is unchanged.")

Wrote 891 / 1026 trials to:
  HRPilot-17_Control/BASELINE5_HRPILOT-17_BOOTH1_250US_10KHZ_8-26-26\BASELINE5_HRPILOT-17_BOOTH1_250US_10KHZ_8-26-26_20260826T101805_cleaned.hrs2

Excluded 135 trial(s). Original file is unchanged.


# Section 8: Convert Conditioning Stage File → .hrs2

Reads any conditioning stage file (`.hrs3`, `.hrs4`, `.hrs5`, `.hrs6`) and writes it as a `.hrs2` (Control Mode) binary file.

**Use case:** a session was recorded using a conditioning stage but the animal was performing control-mode-like behaviour, and you want the data analysed as plain peri-stimulus trials in `Read_H-Reflex_App.ipynb`.

**What is preserved:**
- All raw EMG trial data, sync data, timestamps, stim amplitude, onset indices, background EMG, polarity
- M-wave response, H-wave response, H:M ratio, and M-wave stabilisation fields (if the source file_version ≥ 2)

**What is discarded:**
- Conditioning-specific metadata only: `success_threshold`, `is_success`, `pellet_delivered` / `vns_delivered`, `aux_flag` — these have no equivalent in `.hrs2` and are not needed for EMG analysis.

After writing, add the output file's directory to `RECORDING_DIRS` in `Read_H-Reflex_App.ipynb` with extension `.hrs2`.

In [3]:
# ── Convert Conditioning Stage File → .hrs2 ───────────────────────────────────
# Reads a .hrs3 / .hrs4 / .hrs5 / .hrs6 file and writes it as a .hrs2 binary.
# All EMG trial data is preserved.  Conditioning-only fields are silently ignored.
#
# ── Configuration ─────────────────────────────────────────────────────────────
import os as _os, copy as _copy, math as _math

SOURCE_PATH = (
    "HRPilot-17_Control/BASELINE7_HRPILOT-17_BOOTH1_250US_10KHZ_8-28-26/"
    "BASELINE7_HRPILOT-17_BOOTH1_250US_10KHZ_8-28-26_20260828T102534.hrs4"
)

# Output path — defaults to same directory and stem, extension changed to .hrs2.
# Override this to write elsewhere, e.g.:
#   OUTPUT_PATH = "HRPilot-17_Control/BASELINE1_.../BASELINE1_cleaned.hrs2"
_src_stem, _src_ext = _os.path.splitext(SOURCE_PATH)
OUTPUT_PATH = _src_stem + '.hrs2'

# Stage label written into the output header (shown in notebooks / summaries).
# None = keep whatever is in the source file.
STAGE_NAME = "Control Mode"
STAGE_DESC = None   # None → keep source description unchanged

# ── Reader map — add new extensions here if needed ───────────────────────────
_EXT_TO_READER = {
    '.hrs3': read_hrs3,
    '.hrs4': read_hrs4,
    '.hrs5': read_hrs5,
    '.hrs6': read_hrs6,
}

# ── Validation ────────────────────────────────────────────────────────────────
_ext = _src_ext.lower()
_ok  = True

if _ext not in _EXT_TO_READER:
    print(f"ERROR: unsupported source extension {_src_ext!r}.")
    print(f"       Supported extensions: {list(_EXT_TO_READER.keys())}")
    _ok = False
elif not _os.path.isfile(SOURCE_PATH):
    print(f"ERROR: source file not found:\n  {SOURCE_PATH}")
    _ok = False
elif _os.path.abspath(OUTPUT_PATH) == _os.path.abspath(SOURCE_PATH):
    print("ERROR: OUTPUT_PATH is the same as SOURCE_PATH — choose a different output path.")
    _ok = False

if _ok:
    # ── Read ─────────────────────────────────────────────────────────────────
    print(f"Reading {_src_ext!r}: {SOURCE_PATH}")
    _src_hdr, _src_trials, _src_emg = _EXT_TO_READER[_ext](SOURCE_PATH)
    print(f"  {len(_src_trials)} trials  |  source file_version={_src_hdr.file_version}"
          f"  |  stage: {_src_hdr.stage_name!r}")
    print(f"  sample_rate={_src_hdr.sample_rate} Hz  |  subject: {_src_hdr.subject_id!r}")

    # ── Build output header ───────────────────────────────────────────────────
    _out_hdr = _copy.deepcopy(_src_hdr)
    if STAGE_NAME is not None:
        _out_hdr.stage_name = STAGE_NAME
    if STAGE_DESC is not None:
        _out_hdr.stage_description = STAGE_DESC

    # Choose output file_version:
    #   9  → writes M-wave response fields; read back as CONTROL_MODE_TRIAL (block 6).
    #       Use when source has h_wave_response populated (conditioning fv >= 2).
    #   7  → writes all base trial fields; read back as MH_TRIAL (block 3).
    #       Fallback when h_wave_response is NaN (conditioning fv == 1).
    _first_h = _src_trials[0].h_wave_response if _src_trials else float('nan')
    try:
        _has_h = not _math.isnan(float(_first_h))
    except (TypeError, ValueError):
        _has_h = False
    _TARGET_FV = 9 if _has_h else 7

    # ── Write ─────────────────────────────────────────────────────────────────
    write_hrs2(OUTPUT_PATH, _out_hdr, _src_trials, _src_emg, file_version=_TARGET_FV)

    print(f"\nWrote {len(_src_trials)} trials  →  {OUTPUT_PATH}")
    print(f"  Stage name    : {_out_hdr.stage_name!r}")
    print(f"  File version  : {_TARGET_FV}"
          + ("  (Control Mode — M-wave fields included)" if _TARGET_FV == 9
             else "  (MH-Recruitment compatible — no M-wave response fields)"))
    print(f"\nTo analyse in Read_H-Reflex_App.ipynb, add to RECORDING_DIRS:")
    _out_dir = _os.path.dirname(OUTPUT_PATH) or "."
    print(f'  ("<label>", "{_out_dir}", {_src_hdr.sample_rate}),')


Reading '.hrs4': HRPilot-17_Control/BASELINE7_HRPILOT-17_BOOTH1_250US_10KHZ_8-28-26/BASELINE7_HRPILOT-17_BOOTH1_250US_10KHZ_8-28-26_20260828T102534.hrs4
  826 trials  |  source file_version=4  |  stage: 'S4'
  sample_rate=10000.0 Hz  |  subject: 'UPCP1_HRPILOT-17_BOOTH1_250US_10KHZ_8-28-26'

Wrote 826 trials  →  HRPilot-17_Control/BASELINE7_HRPILOT-17_BOOTH1_250US_10KHZ_8-28-26/BASELINE7_HRPILOT-17_BOOTH1_250US_10KHZ_8-28-26_20260828T102534.hrs2
  Stage name    : 'Control Mode'
  File version  : 9  (Control Mode — M-wave fields included)

To analyse in Read_H-Reflex_App.ipynb, add to RECORDING_DIRS:
  ("<label>", "HRPilot-17_Control/BASELINE7_HRPILOT-17_BOOTH1_250US_10KHZ_8-28-26", 10000.0),
